In [1]:
import cadquery as cq
from cadquery import exporters
import math
from fractions import Fraction
from jupyter_cadquery import *

from jupyter_cadquery.replay import replay, enable_replay, disable_replay

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

versions()

Overwriting auto display for cadquery Workplane and Shape

Versions:
- jupyter_cadquery  4.0.2
- cad_viewer_widget 3.0.2
- open cascade      7.7.2.1



In [2]:
cv = open_viewer("Plate", cad_width=800, height=600)
set_default_viewer("Plate")

In [3]:
# ------------------------------------------------------------
# BRUSH-EVERARD EAVES ASSEMBLY# World coordinates:
#   X = exterior (-) to interior (+)
#   Y = along wall / eave
#   Z = vertical
# ------------------------------------------------------------

# ------------------------------------------------------------
# POST
# ------------------------------------------------------------

post_width = 4.0       # X
post_depth = 6.0       # Y
post_height = 96.0     # Z


# ------------------------------------------------------------
# TOP PLATE
# ------------------------------------------------------------

plate_width = 4.0      # X
plate_depth = 100.0      # Y
plate_height = 6.0     # Z


# ------------------------------------------------------------
# CEILING JOIST
# ------------------------------------------------------------

joist_length = 144.0       # X
joist_thickness = 3.0      # Y
joist_depth = 8.0          # Z

# Joist projects beyond exterior face of top plate
joist_exterior_overhang = 12.0

# Amount the joist is notched down around the top plate
joist_notch_depth = 2.0

notch_clearance = 0.02


# ------------------------------------------------------------
# FALSE PLATE
# ------------------------------------------------------------

false_plate_width = 8.0       # X
false_plate_depth = 100.0       # Y
false_plate_thickness = 1.5   # Z

# Distance from exterior end of ceiling joist to false plate
false_plate_end_offset = 2.0


In [4]:
# ------------------------------------------------------------
# DERIVED PLACEMENT
# ------------------------------------------------------------

# Wall datum:
#   exterior face = X = 0
#   interior      = +X
#   post top      = Z = 0

# Post
post_x0 = 0.0
post_y0 = -post_depth / 2
post_z0 = -post_height

# Top plate
plate_x0 = 0.0
plate_y0 = -plate_depth / 2
plate_z0 = 0.0

# Ceiling joist
#
# The joist projects beyond the exterior face of the wall.
joist_x0 = -joist_exterior_overhang
joist_y0 = -joist_thickness / 2

# Position the joist so the top plate penetrates upward into
# the joist by joist_notch_depth. We'll cut that overlap out next.
joist_bottom_z = plate_height - joist_notch_depth
joist_top_z = joist_bottom_z + joist_depth

# False plate
#
# Located near the exterior end of the ceiling joist and
# resting on top of it.
false_plate_x0 = (
    joist_x0
    + false_plate_end_offset
)

false_plate_y0 = -false_plate_depth / 2
false_plate_z0 = joist_top_z

# ------------------------------------------------------------
# JOINERY
# ------------------------------------------------------------

# Ceiling joist notch
notch_clearance = 0.02

# Post -> top plate tenon
post_tenon_width = 3.0      # X
post_tenon_depth = 2.0      # Y
post_tenon_length = 2.0     # Z into plate

mortise_clearance = 0.02
mortise_extra_depth = 0.25

In [5]:
# ------------------------------------------------------------
# POST
# ------------------------------------------------------------

post = cq.Solid.makeBox(
    post_width,
    post_depth,
    post_height,
    cq.Vector(
        post_x0,
        post_y0,
        post_z0,
    ),
)

In [6]:
# ------------------------------------------------------------
# TOP PLATE
# ------------------------------------------------------------

top_plate = cq.Solid.makeBox(
    plate_width,
    plate_depth,
    plate_height,
    cq.Vector(
        plate_x0,
        plate_y0,
        plate_z0,
    ),
)

In [7]:
# ------------------------------------------------------------
# CEILING JOIST — RAW BLANK
# ------------------------------------------------------------

ceiling_joist_blank = cq.Solid.makeBox(
    joist_length,
    joist_thickness,
    joist_depth,
    cq.Vector(
        joist_x0,
        joist_y0,
        joist_bottom_z,
    ),
)

In [8]:
# ------------------------------------------------------------
# FALSE PLATE
# ------------------------------------------------------------

false_plate = cq.Solid.makeBox(
    false_plate_width,
    false_plate_depth,
    false_plate_thickness,
    cq.Vector(
        false_plate_x0,
        false_plate_y0,
        false_plate_z0,
    ),
)


In [9]:
# ------------------------------------------------------------
# CEILING JOIST NOTCH AROUND TOP PLATE
# ------------------------------------------------------------

joist_notch = cq.Solid.makeBox(
    plate_width + notch_clearance,
    joist_thickness + 2.0,
    joist_notch_depth + 0.25,
    cq.Vector(
        plate_x0 - notch_clearance / 2,
        -(joist_thickness + 2.0) / 2,
        joist_bottom_z,
    ),
)

ceiling_joist = (
    ceiling_joist_blank
    .cut(joist_notch)
    .clean()
)

In [10]:
# ------------------------------------------------------------
# POST TENON
# ------------------------------------------------------------

post_tenon_x0 = (
    post_x0
    + (post_width - post_tenon_width) / 2
)

post_tenon_y0 = (
    post_y0
    + (post_depth - post_tenon_depth) / 2
)

post_tenon = cq.Solid.makeBox(
    post_tenon_width,
    post_tenon_depth,
    post_tenon_length,
    cq.Vector(
        post_tenon_x0,
        post_tenon_y0,
        0.0,
    ),
)

post_with_tenon = (
    post
    .fuse(post_tenon)
    .clean()
)

In [11]:
# ------------------------------------------------------------
# MATCHING TOP-PLATE MORTISE
# ------------------------------------------------------------

mortise_width = (
    post_tenon_width
    + mortise_clearance
)

mortise_depth = (
    post_tenon_depth
    + mortise_clearance
)

mortise_length = (
    post_tenon_length
    + mortise_extra_depth
)

plate_mortise = cq.Solid.makeBox(
    mortise_width,
    mortise_depth,
    mortise_length,
    cq.Vector(
        post_tenon_x0 - mortise_clearance / 2,
        post_tenon_y0 - mortise_clearance / 2,
        0.0,
    ),
)

top_plate_cut = (
    top_plate
    .cut(plate_mortise)
    .clean()
)

In [12]:
show_object(post_with_tenon, name="post")
show_object(top_plate_cut, name="top_plate")
show_object(ceiling_joist, name="ceiling_joist")
show_object(false_plate, name="false_plate")

+
++
+++
c+++
